# GEM中国电源与电网拓扑节点匹配

本notebook使用Global Energy Monitor（GEM）的Global Integrated Power
Tracker（GIPT）单位/阶段级数据，仅提取 `Country/area == "China"` 的中国大陆记录，
检查坐标完整性，并为每条电源记录生成一个可审计的OSM电网候选接入节点。

官方数据页：
<https://globalenergymonitor.org/projects/global-integrated-power-tracker>

官方下载表单要求填写用途并确认CC BY 4.0许可，且不提供稳定的匿名直链。
因此本notebook不会绕过表单；更新数据时请从上面的官方页面下载完整工作簿并放入
`data/`。当前本地文件是官网所列的2026年3月最新版本。

**重要限制：** GEM提供电源经纬度和定位精度，但不提供真实并网变电站或并网电压。
下文结果是用于模型初始化的候选映射，不是经电网企业核实的物理接线关系。

In [1]:
from pathlib import Path
import json

from IPython.core.interactiveshell import InteractiveShell
from IPython.display import Markdown, display
import geopandas as gpd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Wedge
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

InteractiveShell.ast_node_interactivity = "last_expr"
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)

# Data sources and generated output.
gem_download_page = (
    "https://globalenergymonitor.org/projects/"
    "global-integrated-power-tracker#download"
)
gem_file_path = Path("data/Global-Integrated-Power-March-2026-II.xlsx")
storage_source_page = (
    "https://www.sandia.gov/ess-ssl/gesdb/public/projects.html"
)
storage_file_path = Path(
    "data/doe_global_energy_storage_database_2022.json"
)
official_capacity_source = (
    "https://pdf.dfcfw.com/pdf/H3_AP202512311811775347_1.pdf"
)
official_capacity_file_path = Path(
    "data/cec_province_capacity_2023.csv"
)
technical_economic_assumptions_file_path = Path(
    "data/parameter/technical_economic_assumptions.csv"
)
pypsa_costs_file_path = Path(
    "data/parameter/pypsa_technology_data_costs_2025.csv"
)
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

# Analysis scope and spatial matching.
gem_country_area = "China"
gem_statuses = ("operating", "construction")
storage_country_area = "China"
storage_statuses = ("Operational", "Under Construction")
candidate_distance_km = 100
comparison_provinces = (
    "山东省", "江苏省", "新疆维吾尔自治区", "四川省",
)

# Final map settings.
capacity_map_figure_size = (15, 12)
capacity_map_dpi = 300
capacity_map_max_pie_radius_m = 70000

## 1. 校验、读取并汇总电源与储能数据

`Power facilities` 是GEM Global Integrated Power Tracker（GIPT）的统一主表。
本notebook没有另行下载水电工作簿；其中水电记录和 `Technology` 分类由GIPT
整合自Global Hydropower Tracker。根据首个配置cell，仅保留指定国家及
`gem_statuses` 中的状态进入后续统计、空间映射和绘图。

水电按 `Technology` 细分为水库型、径流式、抽水蓄能和未分类水电。组合技术
按“抽蓄优先，其次水库型”归入唯一类别，因此不会重复累计容量。`oil/gas`
只保留 `natural_gas` 和 `other_oil_gas` 两类。
太阳能按GEM `Technology`区分`utility_scale_solar`与`solar_thermal`，
风电区分`onshore_wind`与`offshore_wind`。

其他储能来自
[DOE/Sandia Global Energy Storage Database](https://www.sandia.gov/ess-ssl/gesdb/public/projects.html)，
并剔除DOE中的抽蓄以避免与GEM重复。DOE公开底表可批量复现且带经纬度、
MW和MWh，但版本较旧，不能视为当前中国储能全量。

更完整的中国项目库是CNESA DataLink，字段包含省市、状态、技术、MW/MWh、
接入电压和参与方，但完整数据需要授权；Benchmark、GlobalData和Wood
Mackenzie也提供更大的商业项目库。OSM可补充少量精确位置，但覆盖和容量字段
不完整。因此当前代码继续采用DOE作为开放项目级演示底表。

本block还生成`gem_unit_parameters`：技术规则来自Dispa-SET式机组字段并按
GEM技术和容量区间匹配，成本主体来自PyPSA technology-data；光热和地热的
缺项由NREL ATB补充。所有通用代理都保留`assumption_quality`和来源，不能
解释为中国单台机组实测参数。

In [2]:
if not gem_file_path.exists():
    raise FileNotFoundError(
        f"未找到 {gem_file_path}。请先从 {gem_download_page} "
        "完成官方表单下载，并将完整xlsx文件放入data目录。"
    )
if not storage_file_path.exists():
    raise FileNotFoundError(
        f"未找到 {storage_file_path}。数据源：{storage_source_page}"
    )

_about = pd.read_excel(
    gem_file_path, sheet_name="About", header=None,
    nrows=15, engine="openpyxl",
)
gem_release = str(_about.iloc[0, 0])
_gem_units_all = pd.read_excel(
    gem_file_path, sheet_name="Power facilities", engine="openpyxl"
)

_required_columns = {
    "Type", "Country/area", "Plant / Project name", "Unit / Phase name",
    "Capacity (MW)", "Status", "Technology", "Latitude", "Longitude",
    "Location accuracy", "Subnational unit (state, province)",
    "GEM location ID", "GEM unit/phase ID", "Fuel (combustion only)",
    "Fuel classification (oil/gas only)",
}
_missing_columns = _required_columns.difference(_gem_units_all.columns)
if _missing_columns:
    raise ValueError(f"GEM工作簿缺少字段: {sorted(_missing_columns)}")

_gem_units_china = _gem_units_all[
    _gem_units_all["Country/area"].eq(gem_country_area)
]
gem_units = (
    _gem_units_china[
        _gem_units_china["Status"].isin(gem_statuses)
    ]
    .copy()
    .reset_index(names="gem_source_row")
)
for _column in ["Capacity (MW)", "Latitude", "Longitude"]:
    gem_units[_column] = pd.to_numeric(gem_units[_column], errors="coerce")

gem_units["generation_type"] = gem_units["Type"].replace({
    "utility-scale solar": "utility_scale_solar",
})
_technology = gem_units["Technology"].fillna("").str.lower()
_solar = gem_units["Type"].eq("utility-scale solar")
gem_units.loc[
    _solar & _technology.str.contains("solar thermal"),
    "generation_type",
] = "solar_thermal"
_wind = gem_units["Type"].eq("wind")
gem_units.loc[_wind, "generation_type"] = "onshore_wind"
gem_units.loc[
    _wind & _technology.str.contains("offshore"),
    "generation_type",
] = "offshore_wind"
_hydropower = gem_units["Type"].eq("hydropower")
gem_units.loc[_hydropower, "generation_type"] = (
    "other_hydropower"
)
gem_units.loc[
    _hydropower & _technology.str.contains("run-of-river"),
    "generation_type",
] = "run_of_river_hydropower"
gem_units.loc[
    _hydropower & _technology.str.contains("conventional"),
    "generation_type",
] = "reservoir_hydropower"
gem_units.loc[
    _hydropower & _technology.str.contains("pumped storage"),
    "generation_type",
] = "pumped_storage"

_oil_gas = gem_units["Type"].eq("oil/gas")
_fuel = gem_units["Fuel (combustion only)"].fillna("").str.lower()
_natural_gas = _fuel.str.contains(
    "fossil gas|natural gas|lng|coalbed methane|coal mine methane"
)
gem_units.loc[_oil_gas, "generation_type"] = "other_oil_gas"
gem_units.loc[
    _oil_gas & _natural_gas, "generation_type"
] = "natural_gas"

gem_units_gdf = gpd.GeoDataFrame(
    gem_units,
    geometry=gpd.points_from_xy(
        gem_units["Longitude"], gem_units["Latitude"]
    ),
    crs="EPSG:4326",
)

_location_record_counts = gem_units.groupby("GEM location ID").size()
_accuracy_counts = gem_units["Location accuracy"].fillna(
    "missing"
).value_counts(dropna=False)
_source_rows = [
    ("release", gem_release, "Release label recorded inside the workbook."),
    ("source_file", str(gem_file_path), "Local copy downloaded from the official GEM form."),
    ("global_records", len(_gem_units_all), "All global unit or phase records in the workbook."),
    ("china_records_before_status_filter", len(_gem_units_china), "China records before applying gem_statuses."),
    ("selected_statuses", ", ".join(gem_statuses), "GEM statuses admitted to all downstream analysis."),
    ("selected_china_records", len(gem_units), "China records retained after the status filter."),
    ("china_locations", gem_units["GEM location ID"].nunique(), "Distinct GEM project/location identifiers."),
    ("china_units_or_phases", gem_units["GEM unit/phase ID"].nunique(), "Distinct GEM unit/phase identifiers."),
    ("missing_latitude", gem_units["Latitude"].isna().sum(), "Records missing latitude."),
    ("missing_longitude", gem_units["Longitude"].isna().sum(), "Records missing longitude."),
    ("location_accuracy_exact", _accuracy_counts.get("exact", 0), "Records labelled exact in GEM Location accuracy."),
    ("location_accuracy_non_exact_total", _accuracy_counts.drop(labels="exact", errors="ignore").sum(), "All records not labelled exact in GEM Location accuracy."),
    ("duplicated_unit_phase_ids", gem_units["GEM unit/phase ID"].duplicated().sum(), "Duplicated GEM unit/phase identifiers."),
    ("multi_record_locations", _location_record_counts.gt(1).sum(), "Project/location IDs shared by multiple unit or phase records."),
    ("maximum_records_per_location", _location_record_counts.max(), "Maximum unit or phase records sharing one location ID."),
]
for _accuracy, _count in _accuracy_counts.drop(
    labels="exact", errors="ignore"
).items():
    _accuracy_key = str(_accuracy).strip().lower().replace(" ", "_")
    _source_rows.append((
        f"location_accuracy_non_exact_{_accuracy_key}",
        _count,
        f"Non-exact records in the original GEM category: {_accuracy}.",
    ))
gem_source_summary = pd.DataFrame(
    _source_rows, columns=["metric", "value", "description"]
).set_index("metric")

_type_order = [
    "coal", "natural_gas", "other_oil_gas", "bioenergy", "nuclear",
    "utility_scale_solar", "solar_thermal",
    "onshore_wind", "offshore_wind", "geothermal",
    "reservoir_hydropower", "run_of_river_hydropower",
    "pumped_storage", "other_hydropower",
]
_type_metrics = (
    gem_units.groupby("generation_type", dropna=False)
    .agg(
        records=("GEM unit/phase ID", "size"),
        locations=("GEM location ID", "nunique"),
        capacity_mw=("Capacity (MW)", "sum"),
    )
    .reindex(_type_order, fill_value=0)
)
_accuracy_by_type = pd.crosstab(
    gem_units["Location accuracy"].fillna("missing"),
    gem_units["generation_type"],
).reindex(columns=_type_metrics.index, fill_value=0)
_accuracy_by_type.index = [
    f"location_accuracy: {accuracy}" for accuracy in _accuracy_by_type.index
]
_status_capacity_mw = (
    gem_units.groupby(
        ["Status", "generation_type"], dropna=False
    )["Capacity (MW)"]
    .sum()
    .unstack(fill_value=0)
    .reindex(columns=_type_metrics.index)
)
type_status_summary = pd.concat([
    _type_metrics.T,
    _accuracy_by_type,
    _status_capacity_mw.div(
        _status_capacity_mw.sum(axis=0), axis=1
    ).rename(index=lambda status: f"status: {status}"),
])

with storage_file_path.open(encoding="utf-8") as _file:
    _storage_projects = json.load(_file)
_storage_rows = []
for _project in _storage_projects:
    if str(_project.get("Country", "")).strip() != storage_country_area:
        continue
    _storage_technologies = sorted({
        subsystem.get("Storage Device", {}).get(
            "Technology Mid-Type"
        )
        for subsystem in (_project.get("Subsystems") or [])
        if subsystem.get("Storage Device", {}).get(
            "Technology Mid-Type"
        )
    })
    _storage_rows.append({
        "storage_project_id": _project.get("ID"),
        "storage_project_name": _project.get("Project/Plant Name"),
        "storage_status": _project.get("Status"),
        "power_capacity_mw": pd.to_numeric(
            _project.get("Rated Power (kW)"), errors="coerce"
        ) / 1000,
        "energy_capacity_mwh": pd.to_numeric(
            _project.get("Storage Capacity (kWh)"), errors="coerce"
        ) / 1000,
        "source_province": _project.get("State/Province"),
        "source_city": _project.get("City"),
        "latitude": pd.to_numeric(
            _project.get("Latitude"), errors="coerce"
        ),
        "longitude": pd.to_numeric(
            _project.get("Longitude"), errors="coerce"
        ),
        "source_storage_technology": (
            "; ".join(_storage_technologies) or "unknown"
        ),
    })
storage_units = pd.DataFrame(_storage_rows)
_storage_technology = storage_units[
    "source_storage_technology"
].str.lower()
storage_units["storage_type"] = np.select(
    [
        _storage_technology.str.contains("pumped hydro"),
        _storage_technology.str.contains("lithium"),
        _storage_technology.str.contains("flow battery"),
        _storage_technology.str.contains("compressed air"),
        _storage_technology.str.contains("heat|thermal"),
        _storage_technology.str.contains("sodium"),
        _storage_technology.str.contains("capacitor"),
    ],
    [
        "pumped_storage", "lithium_ion_storage",
        "flow_battery_storage", "compressed_air_storage",
        "thermal_storage", "sodium_battery_storage",
        "capacitor_storage",
    ],
    default="other_storage",
)
storage_units = storage_units.loc[
    ~storage_units["storage_type"].eq("pumped_storage")
    & storage_units["storage_status"].isin(storage_statuses)
].reset_index(drop=True)
storage_units_gdf = gpd.GeoDataFrame(
    storage_units,
    geometry=gpd.points_from_xy(
        storage_units["longitude"], storage_units["latitude"]
    ),
    crs="EPSG:4326",
)
storage_source_summary = (
    storage_units.groupby(
        ["storage_type", "storage_status"], dropna=False
    )
    .agg(
        projects=("storage_project_id", "nunique"),
        power_capacity_mw=("power_capacity_mw", "sum"),
        energy_capacity_mwh=("energy_capacity_mwh", "sum"),
    )
    .sort_index()
)

_count_rows = ["records", "locations", *_accuracy_by_type.index]
_share_rows = type_status_summary.index.difference(
    [*_count_rows, "capacity_mw"]
)
display(Markdown(f"[打开GEM官方下载表单]({gem_download_page})"))
display(gem_source_summary)
display(
    type_status_summary.style
    .format("{:,.0f}", subset=pd.IndexSlice[_count_rows, :])
    .format("{:,.1f}", subset=pd.IndexSlice[["capacity_mw"], :])
    .format("{:.1%}", subset=pd.IndexSlice[_share_rows, :])
)
display(Markdown(f"[打开DOE/Sandia储能数据库]({storage_source_page})"))
display(
    storage_source_summary.style.format({
        "projects": "{:,.0f}",
        "power_capacity_mw": "{:,.1f}",
        "energy_capacity_mwh": "{:,.1f}",
    })
)

# Attach model-ready technical and economic assumptions to each GEM unit.
def match_generation_assumptions(units, rules):
    _rule_ids = pd.Series(pd.NA, index=units.index, dtype="Int64")
    _technology_text = units["Technology"].fillna("").str.lower()
    _capacity = units["Capacity (MW)"]
    for _rule_id, _rule in rules.iterrows():
        _matches = (
            _rule_ids.isna()
            & units["generation_type"].eq(_rule["source_asset_type"])
            & _technology_text.str.contains(
                _rule["source_technology_pattern"], regex=True, na=False
            )
            & _capacity.ge(_rule["capacity_min_mw"])
            & (
                pd.isna(_rule["capacity_max_mw"])
                | _capacity.lt(_rule["capacity_max_mw"])
            )
        )
        _rule_ids.loc[_matches] = _rule_id
    _matched = rules.reindex(_rule_ids).reset_index(drop=True)
    _matched.index = units.index
    return _matched

technical_economic_assumptions = pd.read_csv(
    technical_economic_assumptions_file_path
)
_matched_assumptions = match_generation_assumptions(
    gem_units, technical_economic_assumptions
)
_technical_columns = {
    "source_asset_type": "generation_type",
    "committable": "committable",
    "minimum_output_pu": "minimum_output_pu",
    "ramp_up_pu_per_hour": "ramp_up_pu_per_hour",
    "ramp_down_pu_per_hour": "ramp_down_pu_per_hour",
    "minimum_up_time_h": "minimum_up_time_h",
    "minimum_down_time_h": "minimum_down_time_h",
    "startup_time_h": "startup_time_h",
    "efficiency_override": "efficiency_override",
    "technical_source": "source",
    "technical_quality": "assumption_quality",
    "technical_notes": "notes",
}
_economic_columns = {
    "source_asset_type": "generation_type",
    "pypsa_technology": "pypsa_technology",
    "fuel_technology": "fuel_technology",
    "startup_cost_eur_per_mw": "startup_cost_eur_per_mw",
    "shutdown_cost_eur_per_mw": "shutdown_cost_eur_per_mw",
    "economic_source": "source",
    "economic_quality": "assumption_quality",
    "economic_notes": "notes",
}
gem_unit_parameters = gem_units[[
    "GEM unit/phase ID", "GEM location ID", "Plant / Project name",
    "Unit / Phase name", "generation_type", "Technology",
    "Capacity (MW)", "Status",
]].join(
    _matched_assumptions[list(_technical_columns)]
    .rename(columns=_technical_columns).add_prefix("technical_")
).join(
    _matched_assumptions[list(_economic_columns)]
    .rename(columns=_economic_columns).add_prefix("economic_")
)

_costs = pd.read_csv(pypsa_costs_file_path).pivot_table(
    index="technology", columns="parameter", values="value",
    aggfunc="first",
)
_costs.loc["solar_thermal_nrel", [
    "investment", "FOM", "VOM", "efficiency", "lifetime",
]] = [7912 * 0.92, 74.6 / 7912 * 100, 3.8 * 0.92, 0.38, 30]
_costs.loc["geothermal_nrel", [
    "investment", "FOM", "VOM", "lifetime",
]] = [4828 * 0.92, 129 / 4828 * 100, 0, 30]
_cost_columns = {
    "investment": "investment_eur_per_kw",
    "FOM": "fixed_om_percent_per_year",
    "VOM": "variable_om_eur_per_mwh",
    "efficiency": "efficiency",
    "lifetime": "lifetime_years",
}
gem_unit_parameters = gem_unit_parameters.merge(
    _costs[list(_cost_columns)].rename(columns=_cost_columns),
    left_on="economic_pypsa_technology", right_index=True, how="left",
).merge(
    _costs[["fuel", "CO2 intensity"]].rename(columns={
        "fuel": "fuel_eur_per_mwh_th",
        "CO2 intensity": "co2_t_per_mwh_th",
    }),
    left_on="economic_fuel_technology", right_index=True, how="left",
)
gem_unit_parameters["efficiency"] = gem_unit_parameters[
    "technical_efficiency_override"
].fillna(gem_unit_parameters["efficiency"])
gem_unit_parameters["fixed_om_eur_per_kw_year"] = (
    gem_unit_parameters["investment_eur_per_kw"]
    * gem_unit_parameters["fixed_om_percent_per_year"] / 100
)
gem_unit_parameters["startup_cost_eur"] = (
    gem_unit_parameters["economic_startup_cost_eur_per_mw"]
    * gem_unit_parameters["Capacity (MW)"]
)
_fuel_cost_eur_per_mwh = (
    gem_unit_parameters["fuel_eur_per_mwh_th"]
    / gem_unit_parameters["efficiency"].replace(0, np.nan)
).where(
    gem_unit_parameters["fuel_eur_per_mwh_th"].notna(),
    0.0,
)
gem_unit_parameters["marginal_cost_eur_per_mwh"] = (
    gem_unit_parameters["variable_om_eur_per_mwh"].fillna(0)
    + _fuel_cost_eur_per_mwh
)

parameter_coverage_summary = gem_unit_parameters.groupby(
    "generation_type", dropna=False
).agg(
    records=("GEM unit/phase ID", "size"),
    capacity_mw=("Capacity (MW)", "sum"),
    technical_rules_matched=("technical_generation_type", "count"),
    economic_rules_matched=("economic_generation_type", "count"),
    investment_values=("investment_eur_per_kw", "count"),
    efficiency_values=("efficiency", "count"),
)
display(parameter_coverage_summary)

[打开GEM官方下载表单](https://globalenergymonitor.org/projects/global-integrated-power-tracker#download)

,value,description
metric,,
release,Global Integrated Power Tracker March 2026,Release label recorded inside the workbook.
source_file,data/Global-Integrated-Power-March-2026-II.xlsx,Local copy downloaded from the official GEM form.
global_records,182428,All global unit or phase records in the workbook.
china_records_before_status_filter,39797,China records before applying gem_statuses.
selected_statuses,"operating, construction",GEM statuses admitted to all downstream analysis.
selected_china_records,29125,China records retained after the status filter.
china_locations,23443,Distinct GEM project/location identifiers.
china_units_or_phases,29125,Distinct GEM unit/phase identifiers.
missing_latitude,0,Records missing latitude.


generation_type,coal,natural_gas,other_oil_gas,bioenergy,nuclear,utility_scale_solar,solar_thermal,onshore_wind,offshore_wind,geothermal,reservoir_hydropower,run_of_river_hydropower,pumped_storage,other_hydropower
records,"3,652",599,224,"1,665",91,"14,655",50,"6,814",214,1,601,285,185,89
locations,"1,274",274,108,"1,124",23,"13,776",49,"5,544",181,1,600,285,181,89
capacity_mw,"1,445,382.8","190,337.6","20,699.0","38,299.0","99,414.0","858,682.8","4,915.0","683,876.4","71,746.9",16.0,"295,699.0","84,186.0","236,787.0","3,249.0"
location_accuracy: approximate,79,60,1,363,0,"13,922",45,"3,977",111,1,169,103,147,25
location_accuracy: exact,"3,573",539,223,"1,302",91,733,5,"2,837",103,0,432,182,38,64
status: construction,14.3%,15.4%,7.8%,7.3%,38.7%,28.7%,54.9%,27.3%,36.2%,0.0%,6.1%,72.4%,71.1%,5.9%
status: operating,85.7%,84.6%,92.2%,92.7%,61.3%,71.3%,45.1%,72.7%,63.8%,100.0%,93.9%,27.6%,28.9%,94.1%


[打开DOE/Sandia储能数据库](https://www.sandia.gov/ess-ssl/gesdb/public/projects.html)

,records,capacity_mw,technical_rules_matched,economic_rules_matched,investment_values,efficiency_values
generation_type,,,,,,
bioenergy,1665,38299.00,1665,1665,1665,1665
coal,3652,1445382.80,3652,3652,3652,3652
geothermal,1,16.00,1,1,1,0
natural_gas,599,190337.64,599,599,599,599
nuclear,91,99414.00,91,91,91,91
offshore_wind,214,71746.90,214,214,214,0
onshore_wind,6814,683876.40,6814,6814,6814,0
other_hydropower,89,3249.00,89,89,89,89
other_oil_gas,224,20699.00,224,224,224,224


## 2. 载入并清理OSM电网拓扑

直接运行已验证的 `osm_exp.ipynb`，并原位处理其核心对象。第一步将
`grid_topology`、nodes、branches和station GIS视图限制在最大连通图；
第二步删除 `province_intersection_matched == False` 的station nodes，
NetworkX同步删除其关联branches。剩余station作为GEM电源匹配候选集。


In [3]:
from IPython.utils.capture import capture_output

_required_osm_objects = {
    "grid_topology", "grid_nodes_gdf", "station_nodes_gdf",
    "station_areas_gdf", "grid_branches_gdf", "province_polygons",
}
if not _required_osm_objects.issubset(globals()):
    with capture_output():
        get_ipython().run_line_magic("run", "./osm_exp.ipynb")

In [ ]:
InteractiveShell.ast_node_interactivity = "last_expr"
plt.close("all")

required_topology_objects = {
    "grid_topology", "grid_nodes_gdf", "station_nodes_gdf",
    "station_areas_gdf", "grid_branches_gdf", "province_polygons",
}
missing_topology_objects = required_topology_objects.difference(globals())
if missing_topology_objects:
    raise RuntimeError(
        f"osm_exp.ipynb没有生成: {sorted(missing_topology_objects)}"
    )

if any(
    "node_province_name" not in node_data
    or "province_intersection_matched" not in node_data
    or "in_largest_connected_graph" not in node_data
    for _, node_data in grid_topology.nodes(data=True)
):
    raise RuntimeError("grid_topology node缺少省份或最大连通图属性。")

# First retain the largest connected graph, then remove province-unmatched stations.
grid_topology = grid_topology.subgraph([
    node for node, node_data in grid_topology.nodes(data=True)
    if node_data["in_largest_connected_graph"]
]).copy()
_unmatched_station_uids = {
    node for node, node_data in grid_topology.nodes(data=True)
    if node_data["node_type"] == "station"
    and not node_data["province_intersection_matched"]
}
grid_topology.remove_nodes_from(_unmatched_station_uids)
if not nx.is_connected(grid_topology):
    raise RuntimeError("删除无省份station后，保留的grid_topology不再连通。")

_retained_node_uids = set(grid_topology)
grid_nodes_gdf = grid_nodes_gdf.loc[
    grid_nodes_gdf["node_uid"].isin(_retained_node_uids)
].copy()
grid_branches_gdf = grid_branches_gdf.loc[
    grid_branches_gdf["from_node"].isin(_retained_node_uids)
    & grid_branches_gdf["to_node"].isin(_retained_node_uids)
].copy()
station_nodes_gdf = station_nodes_gdf.loc[
    station_nodes_gdf["node_uid"].isin(_retained_node_uids)
].copy()
station_areas_gdf = station_areas_gdf.loc[
    station_areas_gdf["node_uid"].isin(_retained_node_uids)
].copy()

_unassigned_graph_nodes = sum(
    not node_data["province_intersection_matched"]
    for _, node_data in grid_topology.nodes(data=True)
)
_unassigned_station_nodes = (
    ~station_nodes_gdf["province_intersection_matched"]
).sum()

topology_source_summary = pd.DataFrame(
    {
        "value": [
            min_line_voltage / 1000,
            grid_topology.number_of_nodes(),
            grid_topology.number_of_edges(),
            len(_unmatched_station_uids),
            len(station_nodes_gdf),
            len(grid_branches_gdf),
            grid_branches_gdf["is_cross_province"].sum(),
            _unassigned_graph_nodes,
            _unassigned_station_nodes,
            nx.is_connected(grid_topology),
        ],
        "description": [
            "Minimum OSM line voltage included in the current topology.",
            "Nodes retained after largest-graph and province-match filtering.",
            "Branches retained after largest-graph and province-match filtering.",
            "Province-unmatched station nodes removed with their incident branches.",
            "Remaining station nodes available for generator mapping.",
            "Geographic branch records linked to the retained graph endpoints.",
            "Retained branches whose two mapped endpoint provinces differ.",
            "Remaining junction nodes that do not intersect a province polygon.",
            "Remaining station nodes that do not intersect a province polygon.",
            "Whether the retained NetworkX topology is connected.",
        ],
    },
    index=[
        "minimum_line_voltage_kv", "graph_nodes", "graph_branches",
        "removed_unmatched_province_station_nodes",
        "candidate_station_nodes", "geographic_branches",
        "cross_province_branches", "unassigned_graph_nodes",
        "unassigned_station_nodes", "topology_is_connected",
    ],
)
display(topology_source_summary)

station_candidate_summary = (
    station_nodes_gdf.assign(
        node_province_name=station_nodes_gdf[
            "node_province_name"
        ].fillna("unassigned")
    ).groupby("node_province_name", dropna=False)
    .agg(
        station_nodes=("node_uid", "nunique"),
        substations=(
            "station_osm_type",
            lambda values: values.eq("substation").sum(),
        ),
        converters=(
            "station_osm_type",
            lambda values: values.eq("converter").sum(),
        ),
        multi_voltage_nodes=(
            "node_voltage_levels_kv",
            lambda values: values.map(len).gt(1).sum(),
        ),
        voltage_levels_kv=(
            "node_voltage_levels_kv",
            lambda values: ";".join(map(
                str, sorted({
                    voltage
                    for levels in values
                    for voltage in levels
                })
            )),
        ),
    )
    .sort_values("station_nodes", ascending=False)
)
display(station_candidate_summary)


## 3. 标记电源省份并生成候选变电站映射

GEM电源点先与省界空间连接，再生成唯一的 `mapping_scope`：

1. **`inside_station`：** 电源点与OSM变电站面相交；
2. **`same_province_near`：** 同省且距离不超过阈值；
3. **`same_province_distant`：** 同省但距离超过阈值；
4. **`national_fallback_near`：** 不同省或省份未知，距离不超过阈值；
5. **`national_fallback_distant`：** 不同省或省份未知，且超过阈值。

Location accuracy保留为独立的GEM源数据字段。默认模型纳入站内及near记录，
排除两类distant记录。非抽蓄储能使用相同的省份优先和距离阈值规则。

In [ ]:
_station_columns = [
    "node_uid", "station_uid", "station_name", "station_osm_type",
    "station_osm_voltage", "station_operator", "station_class",
    "node_voltage_levels_kv", "node_province_name",
    "node_longitude", "node_latitude", "geometry",
]

_gem_province_matches = gpd.sjoin(
    gem_units_gdf[["geometry"]],
    province_polygons.to_crs(gem_units_gdf.crs),
    how="left",
    predicate="intersects",
).sort_values("province_name", na_position="last")
_gem_province_matches = _gem_province_matches.loc[
    ~_gem_province_matches.index.duplicated(keep="first")
]
gem_units_gdf["gem_province_name"] = _gem_province_matches[
    "province_name"
].reindex(gem_units_gdf.index)
gem_units_metric = gem_units_gdf.to_crs(metric_crs)

def _nearest_station(units, stations):
    if units.empty or stations.empty:
        return None
    return gpd.sjoin_nearest(
        units,
        stations[_station_columns],
        how="left",
        distance_col="connection_distance_m",
    ).assign(_inside_station=False)

_station_polygons = station_areas_gdf[
    station_areas_gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
][_station_columns]
_inside_matches = (
    gpd.sjoin(
        gem_units_metric,
        _station_polygons,
        how="inner",
        predicate="intersects",
    )
    .assign(connection_distance_m=0.0, _inside_station=True)
    .sort_values(["GEM unit/phase ID", "node_uid"])
    .drop_duplicates("GEM unit/phase ID")
)
_match_parts = [_inside_matches] if not _inside_matches.empty else []
_unmatched_units = gem_units_metric[
    ~gem_units_metric["GEM unit/phase ID"].isin(
        _inside_matches["GEM unit/phase ID"]
    )
]

for _province_name, _units in _unmatched_units.dropna(
    subset=["gem_province_name"]
).groupby("gem_province_name"):
    _matches = _nearest_station(
        _units,
        station_nodes_gdf[
            station_nodes_gdf["node_province_name"].eq(_province_name)
        ],
    )
    if _matches is not None:
        _match_parts.append(_matches)

_matched_ids = (
    set(pd.concat(_match_parts)["GEM unit/phase ID"])
    if _match_parts else set()
)
_fallback_matches = _nearest_station(
    gem_units_metric[
        ~gem_units_metric["GEM unit/phase ID"].isin(_matched_ids)
    ],
    station_nodes_gdf,
)
if _fallback_matches is not None:
    _match_parts.append(_fallback_matches)

gem_grid_matches = gpd.GeoDataFrame(
    pd.concat(_match_parts, ignore_index=True),
    geometry="geometry",
    crs=metric_crs,
).sort_values(
    ["GEM unit/phase ID", "connection_distance_m", "node_uid"]
).drop_duplicates(
    "GEM unit/phase ID"
).reset_index(drop=True)

gem_grid_matches["connection_distance_km"] = (
    gem_grid_matches["connection_distance_m"] / 1000
)
gem_grid_matches["same_province"] = (
    gem_grid_matches["gem_province_name"].notna()
    & gem_grid_matches["node_province_name"].notna()
    & gem_grid_matches["gem_province_name"].eq(
        gem_grid_matches["node_province_name"]
    )
)
gem_grid_matches["candidate_node_voltage_levels_kv"] = (
    gem_grid_matches["node_voltage_levels_kv"].map(
        lambda values: ";".join(map(str, values)) if values else ""
    )
)

_near = gem_grid_matches["connection_distance_km"].le(
    candidate_distance_km
)
gem_grid_matches["mapping_scope"] = np.select(
    [
        gem_grid_matches["_inside_station"],
        gem_grid_matches["same_province"] & _near,
        gem_grid_matches["same_province"],
        _near,
    ],
    [
        "inside_station",
        "same_province_near",
        "same_province_distant",
        "national_fallback_near",
    ],
    default="national_fallback_distant",
)
gem_grid_matches["model_inclusion"] = ~gem_grid_matches[
    "mapping_scope"
].str.endswith("distant")
gem_grid_matches.drop(columns="_inside_station", inplace=True)

if len(gem_grid_matches) != len(gem_units):
    raise AssertionError("候选映射没有保持GEM单位/阶段记录总数。")

_scope_order = [
    "inside_station", "same_province_near", "same_province_distant",
    "national_fallback_near", "national_fallback_distant",
]
mapping_summary = (
    gem_grid_matches.groupby("mapping_scope")
    .agg(
        records=("GEM unit/phase ID", "size"),
        locations=("GEM location ID", "nunique"),
        candidate_station_nodes=("node_uid", "nunique"),
        capacity_mw=("Capacity (MW)", "sum"),
        exact_coordinates=(
            "Location accuracy", lambda values: values.eq("exact").sum()
        ),
        median_distance_km=("connection_distance_km", "median"),
        p95_distance_km=(
            "connection_distance_km",
            lambda values: values.quantile(0.95),
        ),
    )
    .reindex(_scope_order, fill_value=0)
)
mapping_summary["record_share_percent"] = (
    100 * mapping_summary["records"] / len(gem_grid_matches)
)
display(
    mapping_summary.style
    .format("{:,.1f}", subset=["capacity_mw"])
    .format(
        "{:.2f}",
        subset=["median_distance_km", "p95_distance_km"],
    )
    .format("{:.2f}%", subset=["record_share_percent"])
)

_storage_province_matches = gpd.sjoin(
    storage_units_gdf[["geometry"]],
    province_polygons.to_crs(storage_units_gdf.crs),
    how="left",
    predicate="intersects",
).sort_values("province_name", na_position="last")
_storage_province_matches = _storage_province_matches.loc[
    ~_storage_province_matches.index.duplicated(keep="first")
]
storage_units_gdf["storage_province_name"] = (
    _storage_province_matches["province_name"].reindex(
        storage_units_gdf.index
    )
)
storage_units_metric = storage_units_gdf.to_crs(metric_crs)

_storage_match_parts = []
for _province_name, _units in storage_units_metric.dropna(
    subset=["storage_province_name"]
).groupby("storage_province_name"):
    _matches = _nearest_station(
        _units,
        station_nodes_gdf[
            station_nodes_gdf["node_province_name"].eq(_province_name)
        ],
    )
    if _matches is not None:
        _storage_match_parts.append(_matches)

_matched_storage_ids = (
    set(pd.concat(_storage_match_parts)["storage_project_id"])
    if _storage_match_parts else set()
)
_storage_fallback_matches = _nearest_station(
    storage_units_metric[
        ~storage_units_metric["storage_project_id"].isin(
            _matched_storage_ids
        )
    ],
    station_nodes_gdf,
)
if _storage_fallback_matches is not None:
    _storage_match_parts.append(_storage_fallback_matches)

storage_grid_matches = gpd.GeoDataFrame(
    pd.concat(_storage_match_parts, ignore_index=True),
    geometry="geometry",
    crs=metric_crs,
).sort_values(
    ["storage_project_id", "connection_distance_m", "node_uid"]
).drop_duplicates(
    "storage_project_id"
).reset_index(drop=True)
storage_grid_matches["connection_distance_km"] = (
    storage_grid_matches["connection_distance_m"] / 1000
)
storage_grid_matches["same_province"] = (
    storage_grid_matches["storage_province_name"].notna()
    & storage_grid_matches["node_province_name"].notna()
    & storage_grid_matches["storage_province_name"].eq(
        storage_grid_matches["node_province_name"]
    )
)
_storage_near = storage_grid_matches[
    "connection_distance_km"
].le(candidate_distance_km)
storage_grid_matches["mapping_scope"] = np.select(
    [
        storage_grid_matches["same_province"] & _storage_near,
        storage_grid_matches["same_province"],
        _storage_near,
    ],
    [
        "same_province_near",
        "same_province_distant",
        "national_fallback_near",
    ],
    default="national_fallback_distant",
)
storage_grid_matches["model_inclusion"] = ~storage_grid_matches[
    "mapping_scope"
].str.endswith("distant")

if len(storage_grid_matches) != len(storage_units):
    raise AssertionError("候选映射没有保持DOE储能项目总数。")


## 4. 形成模型汇总并与CEC省级装机校核

`node_capacity` 按候选变电站、细分电源类型和状态聚合容量。CEC基准来自
《中国电力统计年鉴—2024》的2023年末省级数据。GEM按照首个cell中的
`gem_statuses` 汇总，因此当前比较的是“在运+在建项目库”与“2023年末在运
行政统计”的差异，不应解释为同一时点的统计误差。

为与CEC大类对齐：水库型、径流式、抽蓄和未分类水电合并为水电；
天然气、其他油气、煤电和生物质合并为火电；核电和地热归入其他。
`selected_province_comparison` 重点展示山东、江苏、新疆和四川，也可在首个
配置cell中修改省份列表。

省级CEC对比使用全部已选状态GEM记录的电源坐标归属省份，不应用候选变电站距离阈值；该阈值只影响节点映射和最终地图。

In [ ]:
_model_role = {
    "coal": "thermal_dispatchable",
    "natural_gas": "thermal_dispatchable",
    "other_oil_gas": "thermal_dispatchable",
    "bioenergy": "thermal_dispatchable",
    "nuclear": "nuclear_dispatchable",
    "utility_scale_solar": "variable_renewable",
    "solar_thermal": "thermal_storage_renewable",
    "onshore_wind": "variable_renewable",
    "offshore_wind": "variable_renewable",
    "geothermal": "dispatchable_renewable",
    "reservoir_hydropower": "reservoir_hydro",
    "run_of_river_hydropower": "run_of_river_hydro",
    "pumped_storage": "pumped_storage",
    "other_hydropower": "hydro",
}
gem_grid_matches["model_role"] = gem_grid_matches[
    "generation_type"
].map(_model_role)

node_capacity = (
    gem_grid_matches[gem_grid_matches["model_inclusion"]].groupby(
        [
            "node_uid", "station_uid", "node_province_name",
            "candidate_node_voltage_levels_kv", "generation_type",
            "Status", "model_role",
        ],
        dropna=False,
    )
    .agg(
        capacity_mw=("Capacity (MW)", "sum"),
        units_or_phases=("GEM unit/phase ID", "nunique"),
        gem_locations=("GEM location ID", "nunique"),
    )
    .reset_index()
    .sort_values(["node_uid", "generation_type", "Status"])
)

storage_mapping_summary = (
    storage_grid_matches.groupby(
        ["storage_type", "mapping_scope"], dropna=False
    )
    .agg(
        projects=("storage_project_id", "nunique"),
        power_capacity_mw=("power_capacity_mw", "sum"),
        energy_capacity_mwh=("energy_capacity_mwh", "sum"),
        median_distance_km=("connection_distance_km", "median"),
    )
    .sort_index()
)

official_province_capacity = pd.read_csv(official_capacity_file_path)
official_province_capacity["other_capacity_mw"] = (
    official_province_capacity["total_capacity_mw"]
    - official_province_capacity[[
        "hydropower_capacity_mw", "thermal_capacity_mw",
        "wind_capacity_mw", "solar_capacity_mw",
    ]].sum(axis=1)
).clip(lower=0)
_official_long = official_province_capacity.melt(
    id_vars=["province_name", "total_capacity_mw"],
    value_vars=[
        "hydropower_capacity_mw", "thermal_capacity_mw",
        "wind_capacity_mw", "solar_capacity_mw",
        "other_capacity_mw",
    ],
    var_name="official_type",
    value_name="official_capacity_mw",
)
_official_long["official_type"] = _official_long[
    "official_type"
].str.removesuffix("_capacity_mw")

_official_type = {
    "reservoir_hydropower": "hydropower",
    "run_of_river_hydropower": "hydropower",
    "pumped_storage": "hydropower",
    "other_hydropower": "hydropower",
    "coal": "thermal",
    "natural_gas": "thermal",
    "other_oil_gas": "thermal",
    "bioenergy": "thermal",
    "onshore_wind": "wind",
    "offshore_wind": "wind",
    "utility_scale_solar": "solar",
    "solar_thermal": "solar",
    "nuclear": "other",
    "geothermal": "other",
}
_gem_province_capacity = (
    gem_grid_matches[
        gem_grid_matches["Status"].isin(gem_statuses)
    ]
    .assign(
        official_type=lambda frame: frame[
            "generation_type"
        ].map(_official_type)
    )
    .groupby(
        ["gem_province_name", "official_type"], dropna=False
    )["Capacity (MW)"]
    .sum()
    .rename("gem_capacity_mw")
    .reset_index()
    .rename(columns={"gem_province_name": "province_name"})
)
province_capacity_validation = _official_long.merge(
    _gem_province_capacity,
    on=["province_name", "official_type"],
    how="left",
).fillna({"gem_capacity_mw": 0})
province_capacity_validation["official_share_percent"] = (
    100 * province_capacity_validation["official_capacity_mw"]
    / province_capacity_validation["total_capacity_mw"]
)
province_capacity_validation["gem_share_percent"] = (
    100 * province_capacity_validation["gem_capacity_mw"]
    / province_capacity_validation.groupby("province_name")[
        "gem_capacity_mw"
    ].transform("sum")
)
province_capacity_validation["share_difference_pp"] = (
    province_capacity_validation["gem_share_percent"]
    - province_capacity_validation["official_share_percent"]
)
province_capacity_validation["capacity_coverage_percent"] = (
    100 * province_capacity_validation["gem_capacity_mw"]
    / province_capacity_validation["official_capacity_mw"]
)

province_validation_summary = (
    province_capacity_validation.groupby("official_type")
    .agg(
        official_capacity_mw=("official_capacity_mw", "sum"),
        gem_capacity_mw=("gem_capacity_mw", "sum"),
        mean_absolute_share_difference_pp=(
            "share_difference_pp", lambda values: values.abs().mean()
        ),
    )
)
province_validation_summary["capacity_coverage_percent"] = (
    100 * province_validation_summary["gem_capacity_mw"]
    / province_validation_summary["official_capacity_mw"]
)

display(node_capacity.sort_values("capacity_mw", ascending=False).head(20))
display(storage_mapping_summary)
display(
    province_validation_summary.style.format({
        "official_capacity_mw": "{:,.0f}",
        "gem_capacity_mw": "{:,.0f}",
        "mean_absolute_share_difference_pp": "{:.2f}",
        "capacity_coverage_percent": (
            lambda value: "-"
            if pd.isna(value) else f"{value:.1f}%"
        ),
    })
)
selected_province_comparison = (
    province_capacity_validation[
        province_capacity_validation["province_name"].isin(
            comparison_provinces
        )
    ][[
        "province_name", "official_type",
        "official_capacity_mw", "gem_capacity_mw",
        "official_share_percent", "gem_share_percent",
        "share_difference_pp", "capacity_coverage_percent",
    ]]
    .sort_values(["province_name", "official_type"])
    .reset_index(drop=True)
)
display(
    selected_province_comparison.style.format({
        "official_capacity_mw": "{:,.0f}",
        "gem_capacity_mw": "{:,.0f}",
        "official_share_percent": "{:.1f}%",
        "gem_share_percent": "{:.1f}%",
        "share_difference_pp": "{:+.1f}",
        "capacity_coverage_percent": (
            lambda value: "-"
            if pd.isna(value) else f"{value:.1f}%"
        ),
    })
)

## 5. 最大连通电网上的节点装机结构

最后一格只绘图并保存PNG。背景沿用OSM notebook的省界、白色线晕和蓝色最大
连通电网样式。饼图面积表示节点总功率容量，扇区表示首个配置cell所选状态下的
细分电源和储能类型。候选映射不等同于电网企业确认的真实并网点。

In [ ]:
_generation_for_plot = (
    gem_grid_matches[
        gem_grid_matches["Status"].isin(gem_statuses)
        & gem_grid_matches["model_inclusion"]
    ]
    .groupby(["node_uid", "generation_type"])["Capacity (MW)"]
    .sum()
    .rename("capacity_mw")
    .reset_index()
    .rename(columns={"generation_type": "capacity_type"})
)
_storage_for_plot = (
    storage_grid_matches[
        storage_grid_matches["storage_status"].isin(storage_statuses)
        & storage_grid_matches["model_inclusion"]
    ]
    .groupby(["node_uid", "storage_type"])["power_capacity_mw"]
    .sum()
    .rename("capacity_mw")
    .reset_index()
    .rename(columns={"storage_type": "capacity_type"})
)
_capacity_for_plot = pd.concat(
    [_generation_for_plot, _storage_for_plot], ignore_index=True
)
_capacity_pivot = _capacity_for_plot.pivot_table(
    index="node_uid",
    columns="capacity_type",
    values="capacity_mw",
    aggfunc="sum",
    fill_value=0,
)
_plot_nodes = station_nodes_gdf.merge(
    _capacity_pivot, left_on="node_uid", right_index=True, how="inner"
)
_capacity_types = list(_capacity_pivot.columns)
_plot_nodes["total_power_capacity_mw"] = _plot_nodes[
    _capacity_types
].sum(axis=1)

_type_colors = {
    "coal": "#3F3F46",
    "natural_gas": "#E67E22",
    "other_oil_gas": "#8C564B",
    "bioenergy": "#4D9221",
    "geothermal": "#B8A600",
    "reservoir_hydropower": "#2166AC",
    "run_of_river_hydropower": "#4393C3",
    "pumped_storage": "#67A9CF",
    "other_hydropower": "#92C5DE",
    "nuclear": "#7B3294",
    "utility_scale_solar": "#F2C94C",
    "solar_thermal": "#E69F00",
    "onshore_wind": "#008B8B",
    "offshore_wind": "#56B4E9",
    "lithium_ion_storage": "#D81B60",
    "flow_battery_storage": "#8E44AD",
    "compressed_air_storage": "#00A6D6",
    "thermal_storage": "#F28E2B",
    "sodium_battery_storage": "#CC79A7",
    "capacitor_storage": "#6B7280",
    "other_storage": "#9CA3AF",
}
_map_provinces = province_polygons.to_crs(metric_crs)
_max_capacity = _plot_nodes["total_power_capacity_mw"].max()
_max_radius_m = capacity_map_max_pie_radius_m

fig, ax = plt.subplots(figsize=capacity_map_figure_size)
_ = _map_provinces.plot(
    ax=ax, facecolor="#FAFAF8", edgecolor="#9CA3AF",
    linewidth=0.30, zorder=1,
)
_ = grid_branches_gdf.plot(
    ax=ax, color="white", linewidth=1.50, alpha=0.90, zorder=2
)
_ = grid_branches_gdf.plot(
    ax=ax, color="#0072B2", linewidth=0.55, alpha=0.72, zorder=3
)
_ = station_nodes_gdf.plot(
    ax=ax, color="#4B5563", markersize=1.3, alpha=0.40, zorder=4
)

for _, _node in _plot_nodes.sort_values(
    "total_power_capacity_mw", ascending=False
).iterrows():
    _radius = _max_radius_m * np.sqrt(
        _node["total_power_capacity_mw"] / _max_capacity
    )
    _angle = 0.0
    for _capacity_type in _capacity_types:
        _share = (
            _node[_capacity_type]
            / _node["total_power_capacity_mw"]
        )
        if _share <= 0:
            continue
        ax.add_patch(Wedge(
            (_node.geometry.x, _node.geometry.y),
            _radius,
            360 * _angle,
            360 * (_angle + _share),
            facecolor=_type_colors.get(_capacity_type, "#9CA3AF"),
            edgecolor="white",
            linewidth=0.20,
            zorder=5,
        ))
        _angle += _share

_present_types = [
    capacity_type for capacity_type in _type_colors
    if capacity_type in _capacity_types
    and _capacity_pivot[capacity_type].sum() > 0
]
_type_legend = ax.legend(
    handles=[
        Patch(
            facecolor=_type_colors[capacity_type],
            edgecolor="none",
            label=capacity_type.replace("_", " "),
        )
        for capacity_type in _present_types
    ],
    loc="lower left",
    frameon=False,
    ncol=2,
    fontsize=7.5,
    title="Selected-status generation and storage power",
)
ax.add_artist(_type_legend)

_size_values = np.geomspace(
    max(_max_capacity / 100, 1), _max_capacity, 3
)
_ = ax.legend(
    handles=[
        Line2D(
            [], [], marker="o", linestyle="none",
            markerfacecolor="none", markeredgecolor="#374151",
            markersize=4 + 8 * np.sqrt(value / _max_capacity),
            label=f"{value:,.0f} MW",
        )
        for value in _size_values
    ],
    loc="lower right",
    frameon=False,
    fontsize=8,
    title="Total power capacity\n(pie area proportional)",
)
_ = ax.set(
    xlim=_map_provinces.total_bounds[[0, 2]],
    ylim=_map_provinces.total_bounds[[1, 3]],
    title=(
        "Selected-status generation and storage assigned to station nodes "
        f"on the {min_line_voltage / 1000:g} kV-and-above "
        "largest connected graph"
    ),
)
_ = ax.set_axis_off()
plt.tight_layout()
fig.savefig(
    output_dir / "gem_grid_connection_candidates.png",
    dpi=capacity_map_dpi,
    bbox_inches="tight",
    facecolor="white",
)
plt.show()